# Phase 9 — Interactive Plotly Visualization (60-class VoteNet)

Same workflow as Phase 8: auto-curate BEST / DENSEST / WORST / TINY-object scenes, render each inline, then batch-export as standalone HTML files.

**Color code:**
- 🟢 **Green** — correct prediction (right class, IoU ≥ threshold with a GT)
- 🟠 **Orange** — right location, wrong class
- 🔴 **Red** — false positive
- ⬛ **Black dashed** — ground truth found
- 🟡 **Yellow dashed** — ground truth MISSED

**Prerequisite:** download `val_viz_60class.pkl` from Kaggle (Cell 9b output), rename to `val_predictions_60class.pkl`, place at:
```
/Users/dosvatsky/3D Object Detection/checkpoints/val_predictions_60class.pkl
```

## Cell 1 — Imports and paths

In [8]:
import sys
from pathlib import Path
import numpy as np

sys.path.insert(0, '/Users/dosvatsky/3D Object Detection/scripts')
from interactive_viz_plotly_v2 import show_scene, load_data, aabb_iou, to_corners

PKL_PATH = '/Users/dosvatsky/3D Object Detection/checkpoints/val_predictions_60class.pkl'
OUT_DIR  = Path('/Users/dosvatsky/3D Object Detection/phase9_viz_output')
OUT_DIR.mkdir(exist_ok=True)

assert Path(PKL_PATH).exists(), f'pkl not found at {PKL_PATH} — download from Kaggle first'
print(f'pkl found: {Path(PKL_PATH).stat().st_size/1e6:.1f} MB')

pkl found: 586.9 MB


## Cell 2 — Load data and class metadata

In [9]:
data = load_data(PKL_PATH)
class_names = data['class_names']
print(f'Total scenes: {len(data["scenes"])}')
print(f'Classes:      {len(class_names)}')

# Class real sizes (meters, longest dim) — for identifying small classes
CLASS_REAL_SIZE = {
    'bed':2.0,'table':1.5,'sofa':2.0,'chair':0.55,'toilet':0.6,'desk':1.4,
    'dresser':1.0,'night_stand':0.55,'bookshelf':0.85,'bathtub':1.6,
    'ammo_box':0.35,'binoculars':0.22,'combat_knife':0.30,'flashlight':0.18,
    'gas_mask':0.28,'hand_grenade':0.12,'helmet':0.28,'magazine':0.18,
    'military_radio':0.30,'pistol':0.22,'rifle':0.95,'rocket_launcher':1.20,
    'shotgun':0.95,'sniper_rifle':1.20,'tactical_backpack':0.55,
    'tactical_vest':0.50,'wire_cutter':0.25,'axe':0.60,'barbed_wire_coil':0.90,
    'baton':0.55,'canteen':0.20,'claymore_mine':0.22,'concrete_barrier':2.00,
    'crossbow':0.75,'duffel_bag':0.80,'entrenching_shovel':0.60,
    'field_telephone':0.30,'first_aid_kit':0.30,'flare_gun':0.25,
    'fuel_drum':0.90,'grenade_launcher':0.75,'hedgehog':1.40,'jerry_can':0.47,
    'machete':0.65,'machine_gun':1.25,'military_boots':0.32,'military_cot':1.90,
    'military_drone':0.90,'military_shield':1.30,'mortar_tube':1.30,
    'night_vision_goggles':0.20,'propane_tank':0.60,'rifle_case':1.20,
    'sandbag':0.65,'smoke_grenade':0.15,'stretcher':2.10,'submachine_gun':0.60,
    'tank_mine':0.33,'tank_shell':0.90,'weapon_rack':1.80,
}

FURNITURE_IDS = set(range(10))   # class indices 0-9 are furniture
SMALL_CLASS_IDS = {i for i, c in enumerate(class_names)
                   if CLASS_REAL_SIZE.get(c, 999) < 0.35}
print(f'small class IDs (< 35cm): {sorted(SMALL_CLASS_IDS)}')

Total scenes: 1500
Classes:      60
small class IDs (< 35cm): [11, 12, 13, 14, 15, 16, 17, 18, 19, 26, 30, 31, 36, 37, 38, 45, 50, 54, 57]


## Cell 3 — Auto-compute per-scene statistics

Walks all 1500 val scenes, matches predictions to GTs, records correctness counts. Takes ~30 sec.

In [10]:
def scene_stats(scene, score_thresh=0.5, match_iou=0.25):
    preds = sorted([p for p in scene['predictions'] if p['score'] >= score_thresh],
                   key=lambda p: -p['score'])
    gts = scene['groundtruths']

    # match predictions to GTs greedily by IoU
    pred_status = [None] * len(preds)
    gt_status   = ['missed'] * len(gts)
    pairs = []
    for i, p in enumerate(preds):
        pc = to_corners(p['box'])
        for j, g in enumerate(gts):
            gc = to_corners(g['box'])
            iou = aabb_iou(pc, gc)
            if iou >= match_iou:
                pairs.append((iou, i, j))
    pairs.sort(reverse=True)
    used_p, used_g = set(), set()
    for iou, i, j in pairs:
        if i in used_p or j in used_g: continue
        used_p.add(i); used_g.add(j)
        if preds[i]['class_id'] == gts[j]['class_id']:
            pred_status[i] = 'correct';  gt_status[j] = 'found'
        else:
            pred_status[i] = 'wrong';    gt_status[j] = 'missed'
    for i, st in enumerate(pred_status):
        if st is None: pred_status[i] = 'fp'

    n_gts = len(gts)
    n_correct = sum(1 for s in pred_status if s == 'correct')
    n_fp      = sum(1 for s in pred_status if s == 'fp')
    n_missed  = sum(1 for s in gt_status if s == 'missed')
    unique_gt_classes = len(set(g['class_id'] for g in gts))
    n_small = sum(1 for g in gts if g['class_id'] in SMALL_CLASS_IDS)
    is_furniture_only = n_gts > 0 and all(g['class_id'] in FURNITURE_IDS for g in gts)
    military_ratio = (sum(1 for g in gts if g['class_id'] not in FURNITURE_IDS)
                      / max(n_gts, 1))
    return {
        'scan_idx': scene['scan_idx'],
        'n_gts': n_gts,
        'n_preds': len(preds),
        'n_correct': n_correct,
        'n_fp': n_fp,
        'n_missed': n_missed,
        'precision': n_correct / max(len(preds), 1),
        'recall':    n_correct / max(n_gts, 1),
        'unique_classes': unique_gt_classes,
        'is_furniture_only': is_furniture_only,
        'military_ratio': military_ratio,
        'n_small': n_small,
    }

print(f'computing stats for {len(data["scenes"])} scenes...')
all_stats = [scene_stats(s) for s in data['scenes']]
print('done')

computing stats for 1500 scenes...
done


## Cell 4 — Curate categorical picks (Phase 8 style)

Picks the single best scene for each showcase category.

In [11]:
def pick(stats, key, reverse=True, where=None):
    filt = [s for s in stats if (where(s) if where else True)]
    if not filt: return None
    return sorted(filt, key=lambda s: s[key], reverse=reverse)[0]

BEST      = pick(all_stats, 'n_correct',      True,  where=lambda s: s['n_gts'] >= 3)
DENSEST   = pick(all_stats, 'n_gts',          True)
DIVERSE   = pick(all_stats, 'unique_classes', True)
WORST     = pick(all_stats, 'n_missed',       True,  where=lambda s: s['n_gts'] >= 3)
FURNITURE = pick(all_stats, 'n_correct',      True,  where=lambda s: s['is_furniture_only'])
MILITARY  = pick(all_stats, 'military_ratio', True,
                 where=lambda s: s['n_gts'] >= 4 and s['military_ratio'] >= 0.75)
TINY      = pick(all_stats, 'n_small',        True,  where=lambda s: s['n_small'] >= 2)

picks = [
    ('BEST performance',        BEST),
    ('DENSEST scene',           DENSEST),
    ('MOST DIVERSE classes',    DIVERSE),
    ('WORST recall (honest)',   WORST),
    ('FURNITURE-only',          FURNITURE),
    ('MILITARY-heavy',          MILITARY),
    ('TINY-object case',        TINY),
]

for name, s in picks:
    if s is None:
        print(f'{name:22s} → no scene matched criteria')
        continue
    print(f'{name:22s} → Scene {s["scan_idx"]:4d}  '
          f'(gt={s["n_gts"]:2d}, correct={s["n_correct"]:2d}, '
          f'missed={s["n_missed"]:2d}, classes={s["unique_classes"]:2d}, '
          f'small={s["n_small"]:2d})')

BEST performance       → Scene   81  (gt=13, correct=12, missed= 1, classes=10, small= 6)
DENSEST scene          → Scene    0  (gt=13, correct=10, missed= 3, classes=13, small= 7)
MOST DIVERSE classes   → Scene    0  (gt=13, correct=10, missed= 3, classes=13, small= 7)
WORST recall (honest)  → Scene 1332  (gt=12, correct= 2, missed=10, classes=12, small= 6)
FURNITURE-only         → no scene matched criteria
MILITARY-heavy         → Scene    3  (gt= 8, correct= 5, missed= 3, classes= 8, small= 5)
TINY-object case       → Scene  289  (gt=12, correct= 8, missed= 4, classes= 9, small=10)


## Cell 5 — Render each categorical pick inline

Runs each curated scene through `show_scene` and displays the interactive 3D widget.

In [12]:
for name, s in picks:
    if s is None: continue
    print(f'\n=== Scene {s["scan_idx"]} — {name} ===')
    fig = show_scene(s['scan_idx'], score_threshold=0.5, top_k=20,
                     match_iou=0.25, pkl_path=PKL_PATH)
    fig.show()


=== Scene 81 — BEST performance ===



=== Scene 0 — DENSEST scene ===



=== Scene 0 — MOST DIVERSE classes ===



=== Scene 1332 — WORST recall (honest) ===



=== Scene 3 — MILITARY-heavy ===



=== Scene 289 — TINY-object case ===


## Cell 6 — Threshold experiments

Same scene at three confidence thresholds — shows how many predictions the model produces at each cutoff.

In [13]:
REFERENCE_IDX = BEST['scan_idx']   # use whichever scene from Cell 4 you prefer

for thresh in [0.3, 0.5, 0.7]:
    label = {'0.3': 'RELAXED (score ≥ 0.3)',
             '0.5': 'STANDARD (score ≥ 0.5)',
             '0.7': 'STRICT (score ≥ 0.7)'}[f'{thresh}']
    print(f'\n=== Scene {REFERENCE_IDX} — {label} ===')
    fig = show_scene(REFERENCE_IDX, score_threshold=thresh, top_k=30,
                     match_iou=0.25, pkl_path=PKL_PATH)
    fig.show()


=== Scene 81 — RELAXED (score ≥ 0.3) ===



=== Scene 81 — STANDARD (score ≥ 0.5) ===



=== Scene 81 — STRICT (score ≥ 0.7) ===


## Cell 7 — Batch export all curated scenes as standalone HTMLs

Writes one `.html` per curated scene (+ threshold variants) to `phase9_viz_output/`. Each file is self-contained and opens in any browser.

In [14]:
curated = []
for name, s in picks:
    if s is None: continue
    tag = name.lower().split()[0].replace('-', '_')
    curated.append((tag, s['scan_idx'], 0.5, name))

# threshold variants of the reference scene
curated += [
    ('relaxed', REFERENCE_IDX, 0.3, f'Scene {REFERENCE_IDX} @ relaxed threshold'),
    ('strict',  REFERENCE_IDX, 0.7, f'Scene {REFERENCE_IDX} @ strict threshold'),
]

for tag, idx, thresh, name in curated:
    fig = show_scene(idx, score_threshold=thresh, top_k=30,
                     match_iou=0.25, pkl_path=PKL_PATH)
    out = OUT_DIR / f'phase9_{tag}_scene{idx:04d}_thr{int(thresh*100):02d}.html'
    fig.write_html(str(out))
    print(f'saved: {out.name}   ({name})')

print(f'\nAll {len(curated)} HTMLs in: {OUT_DIR}')

saved: phase9_best_scene0081_thr50.html   (BEST performance)
saved: phase9_densest_scene0000_thr50.html   (DENSEST scene)
saved: phase9_most_scene0000_thr50.html   (MOST DIVERSE classes)
saved: phase9_worst_scene1332_thr50.html   (WORST recall (honest))
saved: phase9_military_heavy_scene0003_thr50.html   (MILITARY-heavy)
saved: phase9_tiny_object_scene0289_thr50.html   (TINY-object case)
saved: phase9_relaxed_scene0081_thr30.html   (Scene 81 @ relaxed threshold)
saved: phase9_strict_scene0081_thr70.html   (Scene 81 @ strict threshold)

All 8 HTMLs in: /Users/dosvatsky/3D Object Detection/phase9_viz_output


## Cell 8 — Free-form: render any specific scene

Change `SCENE_IDX` to any 0-1499.

In [15]:
SCENE_IDX = 42
SCORE_THRESHOLD = 0.50
TOP_K = 20

fig = show_scene(SCENE_IDX, score_threshold=SCORE_THRESHOLD, top_k=TOP_K,
                 match_iou=0.25, pkl_path=PKL_PATH)
fig.show()

## Cell 9 — Find scenes containing a specific class

Useful for demo hunting: e.g. finding scenes with pistols or grenades to show off small-object detection.

In [16]:
def find_scenes_with_class(name, n=10):
    cls_idx = class_names.index(name)
    matches = []
    for sc in data['scenes']:
        for gt in sc['groundtruths']:
            if gt['class_id'] == cls_idx:
                matches.append(sc['scan_idx']); break
    return matches[:n]

for cls in ['pistol','hand_grenade','flashlight','helmet',
            'jerry_can','concrete_barrier','rifle','fuel_drum']:
    scenes = find_scenes_with_class(cls, n=5)
    print(f'{cls:22s} scenes: {scenes}')

pistol                 scenes: [0, 6, 8, 10, 12]
hand_grenade           scenes: [2, 8, 13, 21, 31]
flashlight             scenes: [9, 17, 18, 20, 21]
helmet                 scenes: [0, 1, 4, 10, 16]
jerry_can              scenes: [2, 9, 10, 14, 30]
concrete_barrier       scenes: [17, 25, 29, 34, 38]
rifle                  scenes: [1, 19, 42, 44, 57]
fuel_drum              scenes: [8, 10, 20, 28, 57]


## Cell 10 — Top-down (bird's-eye) view of any scene

In [17]:
SCENE_IDX = 42

fig = show_scene(SCENE_IDX, score_threshold=0.5, top_k=20,
                 match_iou=0.25, pkl_path=PKL_PATH,
                 view_preset='topdown')
fig.show()